# LeetCode 1580 — Put Boxes Into the Warehouse II bằng Python

Link đề: https://leetcode.com/problems/put-boxes-into-the-warehouse-ii/ · Độ khó: **Medium** · Chủ đề: Array, Greedy, Sorting

> **Ghi chú Premium:** đây là bài Premium (🔒). Đề bài và chữ ký dưới đây được viết theo hiểu biết về bài, lấy từ trang lời giải doocs (https://leetcode.doocs.org/en/lc/1580/). Nếu bạn có tài khoản Premium, hãy đối chiếu lại với đề gốc trên LeetCode.

Skeleton LeetCode:

```python
class Solution:
    def maxBoxesInWarehouse(self, boxes: List[int], warehouse: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

1. Mô hình hoá luật đẩy hộp từ **hai phía** của nhà kho và viết **bộ mô phỏng + tìm kiếm vét cạn** để có đáp án chuẩn cho dữ liệu nhỏ.
2. Hiểu khái niệm **chiều cao hiệu dụng** của mỗi phòng: `eff[i] = min(warehouse[i], max(min tiền tố bên trái, min hậu tố bên phải))` — chiều cao lớn nhất của hộp có thể **tới được** phòng `i`.
3. Biến bài toán thành **ghép cặp hộp ↔ phòng** (hộp ≤ chiều cao hiệu dụng) và giải bằng **sắp xếp + tham lam** (hộp nhỏ nhất vào phòng thấp nhất còn vừa).
4. Giải theo cách khác: **sắp hộp giảm dần + hai con trỏ ở hai đầu nhà kho**, và hiểu vì sao bỏ qua hộp quá to là an toàn.
5. Chứng minh vì sao luôn tồn tại một thứ tự đẩy hộp thực hiện được kế hoạch ghép cặp (các phòng phục vụ từ trái nằm bên trái các phòng phục vụ từ phải).

## 2. Đề bài & ví dụ

Cho hai mảng số nguyên dương `boxes` và `warehouse`: chiều cao các hộp (bề rộng 1 đơn vị) và chiều cao của `n` phòng trong nhà kho. Các phòng đánh số `0` đến `n-1` từ trái sang phải; `warehouse[i]` là chiều cao phòng `i`.

Luật đặt hộp:
- Không được **chồng** hộp lên nhau (mỗi phòng tối đa một hộp).
- Được **sắp xếp lại thứ tự** đưa hộp vào.
- Hộp có thể được đẩy vào từ **cả hai phía** (trái hoặc phải).
- Nếu một phòng thấp hơn hộp thì hộp đó (và các hộp phía sau) bị chặn **trước** phòng đó.

Trả về **số hộp tối đa** có thể đặt vào nhà kho.

**Ví dụ 1:** `boxes = [1,2,2,3,4]`, `warehouse = [3,4,1,2]` → `4`.
Một cách: hộp 1 vào phòng 2 (từ trái hoặc phải), hộp 2 vào phòng 3 (từ phải), hộp 2 vào phòng 1 (từ trái), hộp 3 vào phòng 0 (từ trái).

**Ví dụ 2:** `boxes = [3,5,5,2]`, `warehouse = [2,1,3,4,5]` → `3`.
Không thể đặt cả hai hộp cao 5 vì chỉ có 1 phòng cao ≥ 5.

**Ràng buộc:**
- `n == warehouse.length`
- `1 ≤ boxes.length, warehouse.length ≤ 10⁵`
- `1 ≤ boxes[i], warehouse[i] ≤ 10⁹`

In [1]:
import random
import time
from functools import lru_cache
from math import inf
from typing import List

examples = [([1, 2, 2, 3, 4], [3, 4, 1, 2], 4), ([3, 5, 5, 2], [2, 1, 3, 4, 5], 3)]
for boxes, warehouse, want in examples:
    print(f"boxes={boxes}, warehouse={warehouse} -> mong đợi {want}")

boxes=[1, 2, 2, 3, 4], warehouse=[3, 4, 1, 2] -> mong đợi 4
boxes=[3, 5, 5, 2], warehouse=[2, 1, 3, 4, 5] -> mong đợi 3


## 3. Kiến thức nền

### 3.1 Min tiền tố và min hậu tố

- `left[i] = min(warehouse[0..i-1])` — phòng thấp nhất mà hộp phải đi **qua** khi đẩy từ trái để tới phòng `i` (không tính phòng `i`). Quy ước `left[0] = ∞`.
- `right[i] = min(warehouse[i+1..n-1])`, `right[n-1] = ∞`.

Hai mảng tính trong `O(n)` bằng một lượt mỗi chiều.

In [2]:
warehouse = [3, 4, 1, 2]
n = len(warehouse)
left, right = [inf] * n, [inf] * n
for i in range(1, n):
    left[i] = min(left[i - 1], warehouse[i - 1])
for i in range(n - 2, -1, -1):
    right[i] = min(right[i + 1], warehouse[i + 1])
print("warehouse:", warehouse)
print("left     :", left)
print("right    :", right)

warehouse: [3, 4, 1, 2]
left     : [inf, 3, 3, 1]
right    : [1, 1, 2, inf]


### 3.2 Sắp xếp và tham lam ghép cặp "nhỏ nhất còn vừa"

Bài toán con kinh điển: có các hộp và các "khe" (mỗi khe có sức chứa), mỗi khe chứa tối đa 1 hộp có chiều cao ≤ sức chứa. Muốn ghép **nhiều cặp nhất** ⇒ sắp cả hai tăng dần, duyệt hộp từ nhỏ đến lớn, đưa vào khe **nhỏ nhất còn vừa**.

In [3]:
boxes_s = sorted([4, 1, 2, 2, 3])
slots = sorted([3, 3, 1, 2])
ans = i = 0
pairs = []
for x in boxes_s:
    while i < len(slots) and slots[i] < x:
        i += 1
    if i == len(slots):
        break
    pairs.append((x, slots[i]))
    ans, i = ans + 1, i + 1
print("ghép được", ans, "cặp (hộp, khe):", pairs)

ghép được 4 cặp (hộp, khe): [(1, 1), (2, 2), (2, 3), (3, 3)]


### 3.3 Hai con trỏ ở hai đầu mảng

`l = 0`, `r = n - 1`, mỗi bước tiêu thụ một đầu (`l += 1` hoặc `r -= 1`) và dừng khi `l > r`. Dùng cho cách 3.

In [4]:
arr = [3, 4, 1, 2]
l, r = 0, len(arr) - 1
while l <= r:
    print(f"l={l} (cao {arr[l]}), r={r} (cao {arr[r]})")
    if arr[l] >= arr[r]:
        l += 1
    else:
        r -= 1

l=0 (cao 3), r=3 (cao 2)
l=1 (cao 4), r=3 (cao 2)
l=2 (cao 1), r=3 (cao 2)
l=2 (cao 1), r=2 (cao 1)


### 3.4 Bitmask + `lru_cache` cho tìm kiếm vét cạn

Trạng thái của nhà kho = tập phòng đã có hộp (bitmask `mask`) + bộ hộp còn lại (tuple đã sắp). `functools.lru_cache` ghi nhớ kết quả mỗi trạng thái để không tính lại.

In [5]:
@lru_cache(maxsize=None)
def count_subsets_sum(mask_bits, target):
    # ví dụ minh hoạ memo: số tập con của {1..mask_bits} có tổng = target
    if mask_bits == 0:
        return 1 if target == 0 else 0
    return count_subsets_sum(mask_bits - 1, target) + count_subsets_sum(mask_bits - 1, target - mask_bits)

print("số tập con của {1..10} có tổng 15:", count_subsets_sum(10, 15))
print(count_subsets_sum.cache_info())

số tập con của {1..10} có tổng 15: 20
CacheInfo(hits=185, misses=296, maxsize=None, currsize=296)


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 1: `warehouse = [3, 4, 1, 2]`.

**Hộp cao `h` tới được phòng `i` từ trái** ⇔ mọi phòng `0..i` cao ≥ `h` ⇔ `h ≤ min(warehouse[0..i])`.
**Từ phải** ⇔ `h ≤ min(warehouse[i..n-1])`.

| phòng i | cao | min trái (gồm i) | min phải (gồm i) | hộp lớn nhất tới được = max(hai cột) |
|---|---|---|---|---|
| 0 | 3 | 3 | 1 | **3** |
| 1 | 4 | 3 | 1 | **3** |
| 2 | 1 | 1 | 1 | **1** |
| 3 | 2 | 1 | 2 | **2** |

Gọi giá trị cột cuối là **chiều cao hiệu dụng** `eff = [3, 3, 1, 2]`. Giờ bài toán giống như có 4 phòng "thẳng đứng độc lập" với sức chứa `[3,3,1,2]`: ghép hộp `[1,2,2,3,4]` vào ⇒ `1→1`, `2→2`, `2→3`, `3→3`, hộp `4` không vừa ⇒ **4 hộp**. ✔

Câu hỏi còn lại: có luôn tồn tại **thứ tự đẩy** thực hiện được việc ghép này không (hộp này không chặn đường hộp khác)? Phần 10 sẽ chứng minh là có.

In [6]:
def effective_heights(warehouse):
    n = len(warehouse)
    left, right = [inf] * n, [inf] * n
    for i in range(1, n):
        left[i] = min(left[i - 1], warehouse[i - 1])
    for i in range(n - 2, -1, -1):
        right[i] = min(right[i + 1], warehouse[i + 1])
    return [min(warehouse[i], max(left[i], right[i])) for i in range(n)]

for _, wh, _ in examples:
    print("warehouse =", wh, "-> eff =", effective_heights(wh))

warehouse = [3, 4, 1, 2] -> eff = [3, 3, 1, 2]
warehouse = [2, 1, 3, 4, 5] -> eff = [2, 1, 3, 4, 5]


## 5. Cách 1 — Brute Force / baseline: mô phỏng + tìm kiếm vét cạn

**Mô phỏng đúng luật:** khi đẩy hộp cao `h` từ trái, hộp trượt sang phải **xa nhất có thể**: qua các phòng còn trống và cao ≥ `h`; dừng trước phòng đã có hộp, phòng thấp hơn `h`, hoặc cuối kho. Nếu ngay phòng đầu tiên đã không vào được thì không đặt được hộp từ phía đó. Từ phải tương tự.

**Vét cạn:** ở mỗi trạng thái, thử mọi hộp (mỗi giá trị chiều cao một lần) × hai phía, hoặc dừng. Dùng `lru_cache` ghi nhớ. Chỉ chạy được với dữ liệu rất nhỏ — dùng làm **oracle**.

In [7]:
def slide(mask, warehouse, h, side):
    """Trả về phòng mà hộp cao h dừng lại khi đẩy từ side ('L'/'R'), hoặc None nếu không vào được."""
    n = len(warehouse)
    if side == "L":
        pos = -1
        while pos + 1 < n and not (mask >> (pos + 1)) & 1 and warehouse[pos + 1] >= h:
            pos += 1
        return pos if pos >= 0 else None
    pos = n
    while pos - 1 >= 0 and not (mask >> (pos - 1)) & 1 and warehouse[pos - 1] >= h:
        pos -= 1
    return pos if pos < n else None

def brute_force(boxes: List[int], warehouse: List[int]) -> int:
    wh = tuple(warehouse)

    @lru_cache(maxsize=None)
    def go(mask, rem):
        best = 0
        for h in set(rem):
            rest = list(rem)
            rest.remove(h)
            rest = tuple(rest)
            for side in "LR":
                p = slide(mask, wh, h, side)
                if p is not None:
                    best = max(best, 1 + go(mask | (1 << p), rest))
        return best

    return go(0, tuple(sorted(boxes)))

for boxes, wh, want in examples:
    print(boxes, wh, "->", brute_force(boxes, wh), "| mong đợi", want)

[1, 2, 2, 3, 4] [3, 4, 1, 2] -> 4 | mong đợi 4
[3, 5, 5, 2] [2, 1, 3, 4, 5] -> 3 | mong đợi 3


**Độ phức tạp:** số trạng thái lên tới `2ⁿ ×` (số tập con của hộp) — hàm mũ. Chỉ dùng cho `n, m ≤ 7`.

**Nút thắt:** ta đang thử cả **thứ tự** và **phía** đẩy. Nhận xét then chốt: đáp án chỉ phụ thuộc vào **hộp nào vào phòng nào**, và việc một hộp có vào được phòng `i` hay không chỉ phụ thuộc vào chiều cao hiệu dụng `eff[i]`.

In [8]:
rng = random.Random(1)
boxes_t = [rng.randint(1, 6) for _ in range(7)]
wh_t = [rng.randint(1, 6) for _ in range(7)]
t0 = time.perf_counter()
r = brute_force(boxes_t, wh_t)
print(f"n=m=7: {r}, mất {time.perf_counter() - t0:.3f}s (n=m=12 sẽ không chạy nổi)")

n=m=7: 6, mất 0.000s (n=m=12 sẽ không chạy nổi)


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — chiều cao hiệu dụng.** Hộp cao `h` có thể (trong kho trống) đến phòng `i` ⇔ `h ≤ L(i) = min(wh[0..i])` hoặc `h ≤ R(i) = min(wh[i..n-1])` ⇔ `h ≤ eff[i] = max(L(i), R(i))`. Viết theo `left/right` không gồm `i`: `eff[i] = min(wh[i], max(left[i], right[i]))`.

**Insight 2 — hai phía không giẫm chân nhau.** `L(i)` **không tăng** theo `i`, `R(i)` **không giảm**. Nên có điểm `p` sao cho phòng `i ≤ p` có `L(i) ≥ R(i)` (phục vụ từ trái), phòng `i > p` có `R(i) > L(i)` (phục vụ từ phải). Các phòng phục vụ từ trái nằm hết bên trái các phòng phục vụ từ phải ⇒ đẩy hộp từ hai phía không cản nhau.

**Insight 3 — ghép cặp.** Bài toán còn lại: ghép nhiều cặp nhất `(hộp, phòng)` với `hộp ≤ eff[phòng]`. Tham lam: hộp nhỏ nhất vào phòng có `eff` nhỏ nhất còn vừa.

**Insight 4 — góc nhìn hai con trỏ.** Phòng có thể dùng nhất thời luôn là phòng **sát cửa** ở hai đầu. Đưa hộp **lớn nhất trước**: nếu vừa đầu trái thì đặt ở đầu trái, nếu không thì thử đầu phải; nếu không vừa cả hai thì hộp này **không bao giờ** đặt được (các phòng bên trong chỉ có thể thấp hơn về mặt "tới được") ⇒ bỏ qua.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Chiều cao hiệu dụng + sắp xếp + tham lam (lời giải doocs)

**Bước 1:** tính `left`, `right` như phần 3.1.

**Bước 2:** `warehouse[i] = min(warehouse[i], max(left[i], right[i]))` — ghi đè thành chiều cao hiệu dụng (doocs ghi đè trực tiếp; trong notebook ta làm trên bản sao để không đổi dữ liệu test).

**Bước 3:** sắp `boxes` và `eff` tăng dần; với mỗi hộp `x`, bỏ qua các phòng có `eff < x`; nếu còn phòng thì đặt, tăng cả hai.

In [9]:
def max_boxes_eff(boxes: List[int], warehouse: List[int]) -> int:
    eff = sorted(effective_heights(warehouse))
    b = sorted(boxes)
    n = len(eff)
    ans = i = 0
    for x in b:
        while i < n and eff[i] < x:
            i += 1
        if i == n:
            break
        ans, i = ans + 1, i + 1
    return ans

for boxes, wh, want in examples:
    print(boxes, wh, "->", max_boxes_eff(boxes, wh), "| mong đợi", want)

[1, 2, 2, 3, 4] [3, 4, 1, 2] -> 4 | mong đợi 4
[3, 5, 5, 2] [2, 1, 3, 4, 5] -> 3 | mong đợi 3


Code nộp LeetCode cho cách 2 (đúng như doocs, phần 9):

```python
class Solution:
    def maxBoxesInWarehouse(self, boxes: List[int], warehouse: List[int]) -> int:
        n = len(warehouse)
        left = [0] * n
        right = [0] * n
        left[0] = right[-1] = inf
        for i in range(1, n):
            left[i] = min(left[i - 1], warehouse[i - 1])
        for i in range(n - 2, -1, -1):
            right[i] = min(right[i + 1], warehouse[i + 1])
        for i in range(n):
            warehouse[i] = min(warehouse[i], max(left[i], right[i]))
        boxes.sort()
        warehouse.sort()
        ans = i = 0
        for x in boxes:
            while i < n and warehouse[i] < x:
                i += 1
            if i == n:
                break
            ans, i = ans + 1, i + 1
        return ans
```

**Độ phức tạp:** `O(n log n + m log m)` thời gian (sắp xếp), `O(n)` bộ nhớ phụ cho `left`, `right` (với `m = len(boxes)`).

#### Biến thể nhỏ của cách 2: ghép từ hộp lớn nhất

Sắp hộp và `eff` **giảm dần**; con trỏ `j` trên `eff`: với hộp `x` lớn nhất còn lại, nếu `eff[j] ≥ x` thì đặt và `j += 1`, ngược lại bỏ hộp (hộp quá to cho phòng tốt nhất còn lại thì không vừa phòng nào). Kết quả giống hệt — đây là cầu nối sang cách 3.

In [10]:
def max_boxes_eff_desc(boxes, warehouse):
    eff = sorted(effective_heights(warehouse), reverse=True)
    j = ans = 0
    for x in sorted(boxes, reverse=True):
        if j == len(eff):
            break
        if eff[j] >= x:
            ans += 1
            j += 1
    return ans

for boxes, wh, want in examples:
    print(boxes, wh, "->", max_boxes_eff_desc(boxes, wh), "| mong đợi", want)

[1, 2, 2, 3, 4] [3, 4, 1, 2] -> 4 | mong đợi 4
[3, 5, 5, 2] [2, 1, 3, 4, 5] -> 3 | mong đợi 3


### 7.2 Cách 3 — Hộp giảm dần + hai con trỏ ở hai đầu kho

Không cần mảng `left/right`. Duy trì `l`, `r` = phòng trống **ngoài cùng** bên trái / bên phải (các phòng ngoài hơn đã được lấp hoặc bị bỏ qua).

Với hộp `x` (từ lớn đến nhỏ):
- nếu `x ≤ warehouse[l]`: đặt hộp vào phòng `l` (đẩy từ trái, phòng này sẽ là phòng "ngoài cùng" kế tiếp được lấp), `l += 1`;
- elif `x ≤ warehouse[r]`: đặt vào phòng `r`, `r -= 1`;
- else: bỏ hộp `x`.

Dừng khi `l > r` (hết phòng).

*Trực giác:* hộp lớn nhất đến trước nên chỉ cần phòng ngoài cùng đủ cao; các hộp đến sau nhỏ hơn hoặc bằng nên luôn chui **qua** được những phòng đã chứa hộp lớn hơn? Không — chúng không chui qua phòng **có hộp**. Nhưng ở đây ta hiểu ngược: kế hoạch đặt hộp lớn vào phòng ngoài và hộp nhỏ hơn vào phòng sâu hơn; khi **thực sự đẩy**, ta đẩy hộp nhỏ (phòng sâu) **trước**, hộp lớn (phòng ngoài) sau. Hộp nhỏ đi qua các phòng ngoài cao ≥ hộp lớn ≥ nó.

In [11]:
def max_boxes_two_pointers(boxes: List[int], warehouse: List[int]) -> int:
    l, r = 0, len(warehouse) - 1
    ans = 0
    for x in sorted(boxes, reverse=True):
        if l > r:
            break
        if x <= warehouse[l]:
            l += 1
            ans += 1
        elif x <= warehouse[r]:
            r -= 1
            ans += 1
    return ans

for boxes, wh, want in examples:
    print(boxes, wh, "->", max_boxes_two_pointers(boxes, wh), "| mong đợi", want)

[1, 2, 2, 3, 4] [3, 4, 1, 2] -> 4 | mong đợi 4
[3, 5, 5, 2] [2, 1, 3, 4, 5] -> 3 | mong đợi 3


Code nộp LeetCode cho cách 3:

```python
class Solution:
    def maxBoxesInWarehouse(self, boxes: List[int], warehouse: List[int]) -> int:
        l, r = 0, len(warehouse) - 1
        ans = 0
        for x in sorted(boxes, reverse=True):
            if l > r:
                break
            if x <= warehouse[l]:
                l += 1
                ans += 1
            elif x <= warehouse[r]:
                r -= 1
                ans += 1
        return ans
```

**Độ phức tạp:** `O(m log m + n)` thời gian, `O(m)` cho bản sắp của `boxes` (hoặc `O(1)` nếu sắp tại chỗ).

Vì sao bỏ hộp `x` khi không vừa cả `warehouse[l]` lẫn `warehouse[r]` là an toàn? Mọi phòng trống `k ∈ [l, r]` đều nằm **sau** phòng `l` (nhìn từ trái) và sau phòng `r` (nhìn từ phải). Hộp `x` muốn tới `k` phải đi qua `l` hoặc `r` — cả hai đều thấp hơn `x`. Các phòng ngoài `[l, r]` đã được dành cho hộp lớn hơn hoặc bằng `x`; đổi chỗ không làm tăng số hộp (lập luận trao đổi ở phần 10).

## 8. TRACE từng bước

**Cách 2** với ví dụ 1: `warehouse = [3,4,1,2]`, `left = [∞,3,3,1]`, `right = [1,1,2,∞]`, `eff = [3,3,1,2]` → sắp: `[1,2,3,3]`; `boxes` sắp: `[1,2,2,3,4]`.

| hộp x | i trước | bỏ qua eff < x | phòng dùng | ans |
|---|---|---|---|---|
| 1 | 0 | — | eff=1 | 1 |
| 2 | 1 | — | eff=2 | 2 |
| 2 | 2 | — | eff=3 | 3 |
| 3 | 3 | — | eff=3 | 4 |
| 4 | 4 | hết phòng | dừng | **4** |

**Cách 3** với ví dụ 2: `warehouse = [2,1,3,4,5]`, hộp giảm dần `[5,5,3,2]`.

| hộp | l | r | wh[l] | wh[r] | hành động |
|---|---|---|---|---|---|
| 5 | 0 | 4 | 2 | 5 | đặt phải, r=3 |
| 5 | 0 | 3 | 2 | 4 | bỏ |
| 3 | 0 | 3 | 2 | 4 | đặt phải, r=2 |
| 2 | 0 | 2 | 2 | 3 | đặt trái, l=1 |
| | | | | | kết quả **3** |

In [12]:
def trace_eff(boxes, warehouse):
    n = len(warehouse)
    left, right = [inf] * n, [inf] * n
    for i in range(1, n):
        left[i] = min(left[i - 1], warehouse[i - 1])
    for i in range(n - 2, -1, -1):
        right[i] = min(right[i + 1], warehouse[i + 1])
    eff = [min(warehouse[i], max(left[i], right[i])) for i in range(n)]
    print("warehouse:", warehouse)
    print("left     :", left)
    print("right    :", right)
    print("eff      :", eff)
    e, b = sorted(eff), sorted(boxes)
    print("eff sắp  :", e, "| boxes sắp:", b)
    ans = i = 0
    for x in b:
        skipped = []
        while i < n and e[i] < x:
            skipped.append(e[i])
            i += 1
        if i == n:
            print(f"  hộp {x}: bỏ qua {skipped}, hết phòng -> dừng")
            break
        print(f"  hộp {x}: bỏ qua {skipped}, đặt vào phòng eff={e[i]}")
        ans, i = ans + 1, i + 1
    print("kết quả:", ans)
    return ans

trace_eff([1, 2, 2, 3, 4], [3, 4, 1, 2])

warehouse: [3, 4, 1, 2]
left     : [inf, 3, 3, 1]
right    : [1, 1, 2, inf]
eff      : [3, 3, 1, 2]
eff sắp  : [1, 2, 3, 3] | boxes sắp: [1, 2, 2, 3, 4]
  hộp 1: bỏ qua [], đặt vào phòng eff=1
  hộp 2: bỏ qua [], đặt vào phòng eff=2
  hộp 2: bỏ qua [], đặt vào phòng eff=3
  hộp 3: bỏ qua [], đặt vào phòng eff=3
  hộp 4: bỏ qua [], hết phòng -> dừng
kết quả: 4


4

In [13]:
def trace_two_pointers(boxes, warehouse):
    l, r, ans = 0, len(warehouse) - 1, 0
    for x in sorted(boxes, reverse=True):
        if l > r:
            print(f"  hộp {x}: hết phòng")
            break
        if x <= warehouse[l]:
            act = f"đặt vào phòng {l} (trái)"
            l += 1
            ans += 1
        elif x <= warehouse[r]:
            act = f"đặt vào phòng {r} (phải)"
            r -= 1
            ans += 1
        else:
            act = "không vừa hai đầu -> bỏ"
        print(f"  hộp {x}: {act:<24} | l={l} r={r} ans={ans}")
    return ans

print("kết quả:", trace_two_pointers([3, 5, 5, 2], [2, 1, 3, 4, 5]))

  hộp 5: đặt vào phòng 4 (phải)   | l=0 r=3 ans=1
  hộp 5: không vừa hai đầu -> bỏ  | l=0 r=3 ans=1
  hộp 3: đặt vào phòng 3 (phải)   | l=0 r=2 ans=2
  hộp 2: đặt vào phòng 0 (trái)   | l=1 r=2 ans=3
kết quả: 3


## 9. Code nộp LeetCode

Đúng tên phương thức theo doocs: `maxBoxesInWarehouse(self, boxes, warehouse)`. Lưu ý bản này **sửa tại chỗ** `boxes` và `warehouse` (giống doocs) — khi test trong notebook ta truyền bản sao.

In [14]:
class Solution:
    def maxBoxesInWarehouse(self, boxes: List[int], warehouse: List[int]) -> int:
        n = len(warehouse)
        left = [0] * n
        right = [0] * n
        left[0] = right[-1] = inf
        for i in range(1, n):
            left[i] = min(left[i - 1], warehouse[i - 1])
        for i in range(n - 2, -1, -1):
            right[i] = min(right[i + 1], warehouse[i + 1])
        for i in range(n):
            warehouse[i] = min(warehouse[i], max(left[i], right[i]))
        boxes.sort()
        warehouse.sort()
        ans = i = 0
        for x in boxes:
            while i < n and warehouse[i] < x:
                i += 1
            if i == n:
                break
            ans, i = ans + 1, i + 1
        return ans

def solution_copy(boxes, warehouse):
    return Solution().maxBoxesInWarehouse(list(boxes), list(warehouse))

for boxes, wh, want in examples:
    print(boxes, wh, "->", solution_copy(boxes, wh), "| mong đợi", want)

[1, 2, 2, 3, 4] [3, 4, 1, 2] -> 4 | mong đợi 4
[3, 5, 5, 2] [2, 1, 3, 4, 5] -> 3 | mong đợi 3


## 10. Vì sao thuật toán đúng

**(a) Cận trên.** Trong mọi cách đặt, hộp ở phòng `i` phải đi qua (từ một phía) toàn bộ các phòng giữa cửa và `i`, gồm cả phòng `i` ⇒ `hộp ≤ L(i)` hoặc `hộp ≤ R(i)` ⇒ `hộp ≤ eff[i]`. Vậy mọi cách đặt hợp lệ là một **ghép cặp** hộp–phòng với ràng buộc `hộp ≤ eff`. Số hộp ≤ ghép cặp lớn nhất.

**(b) Tham lam cho ghép cặp lớn nhất.** Chuẩn: duyệt hộp tăng dần, đưa vào phòng `eff` nhỏ nhất còn vừa. Lập luận trao đổi: nếu lời giải tối ưu đặt hộp nhỏ nhất `x` vào phòng khác (hoặc không đặt), ta có thể hoán đổi để `x` vào phòng nhỏ nhất vừa mà không giảm số cặp.

**(c) Thực hiện được.** Cho một ghép cặp. Chọn `p` = chỉ số lớn nhất có `L(p) ≥ R(p)` (`L` không tăng, `R` không giảm ⇒ `{i : L(i) ≥ R(i)}` là một tiền tố). Phòng `i ≤ p` có `eff[i] = L(i)`, phòng `i > p` có `eff[i] = R(i)`.
- Bên trái: đẩy hộp từ trái theo thứ tự phòng **xa cửa nhất trước** (`p, p-1, …, 0`), chỉ đẩy vào các phòng được chọn. Khi đẩy hộp cho phòng `i`, mọi phòng `< i` còn trống và cao ≥ `L(i) ≥ hộp` ⇒ hộp đi tới được `i` và ta dừng nó ở đó.
- Bên phải: đối xứng, theo thứ tự `p+1, …, n-1` (xa cửa phải nhất trước).
- Hai bên không cản nhau vì đường đi của bên trái nằm trong `[0, p]`, bên phải trong `[p+1, n-1]`.

*Ghi chú về luật "trượt xa nhất":* lập luận trên coi như ta dừng được hộp tại phòng mong muốn. Nếu hộp buộc phải trượt tiếp tới phòng trống sâu hơn thì nó chỉ chiếm một phòng sâu hơn mà nó vẫn vừa; bằng lập luận trao đổi có thể chỉ ra số hộp tối đa không đổi. Bộ test ở phần 12 đối chiếu cả ba cách với **mô phỏng trượt-xa-nhất** (cách 1) trên 1500 bộ ngẫu nhiên, và bài tập 5 kiểm tra kế hoạch theo luật "dừng tại phòng đích".

**(d) Cách 3.** Hộp lớn nhất `x`: nếu vừa phòng ngoài cùng nào đó, đặt nó ở đó không làm hại (phòng ngoài cùng là phòng "dễ tới nhất" nhưng cũng là phòng chặn mọi phòng sau nó — đặt hộp **lớn nhất** vào phòng chặn là tốt nhất vì các hộp còn lại đều nhỏ hơn, đi qua được khi đẩy trước). Nếu không vừa cả hai đầu thì `x > eff[k]` với mọi phòng trống `k` ⇒ bỏ an toàn.

## 11. Phân tích Time / Space

Gọi `n = len(warehouse)`, `m = len(boxes)`.

| Cách | Time | Space phụ |
|---|---|---|
| 1. Mô phỏng + vét cạn | hàm mũ (`O(2ⁿ · 2ᵐ · m · n)`) | hàm mũ (memo) |
| 2. `eff` + sắp + tham lam | `O(n log n + m log m)` | `O(n)` |
| 2'. `eff` giảm dần | `O(n log n + m log m)` | `O(n)` |
| 3. Hộp giảm dần + hai con trỏ | `O(m log m + n)` | `O(1)`–`O(m)` |

Nút cổ chai ở cả hai cách nhanh là **sắp xếp**. Không có khác biệt average/worst.

## 12. Bộ test

In [15]:
ALL = [brute_force, max_boxes_eff, max_boxes_eff_desc, max_boxes_two_pointers, solution_copy]

for f in ALL:
    for boxes, wh, want in examples:
        assert f(list(boxes), list(wh)) == want, (f.__name__, boxes, wh)
print("Ví dụ đề: OK")

edge = [
    ([1], [1], 1),                         # nhỏ nhất
    ([2], [1], 0),                         # hộp không vừa
    ([5, 5, 5], [5], 1),                   # một phòng
    ([1, 1, 1], [5, 1, 5], 3),             # phòng giữa thấp nhưng hộp 1 vẫn vào
    ([3, 3, 3], [5, 1, 5], 2),             # phòng giữa chỉ vừa 1, hai đầu vừa 3
    ([4, 4], [4, 9, 9, 4], 2),             # hai cửa thấp chặn hộp lớn hơn 4
    ([1, 2, 3, 4], [4, 3, 2, 1], 4),       # giảm dần: đẩy từ trái được hết
]
for f in ALL:
    for boxes, wh, want in edge:
        assert f(list(boxes), list(wh)) == want, (f.__name__, boxes, wh, f(list(boxes), list(wh)), want)
print("Edge case: OK")

Ví dụ đề: OK
Edge case: OK


In [16]:
# Ngẫu nhiên seed cố định, đối chiếu với mô phỏng vét cạn (dữ liệu nhỏ)
rng = random.Random(1580)
for _ in range(1500):
    n, m = rng.randint(1, 6), rng.randint(1, 6)
    H = rng.choice([3, 6, 10])
    wh = [rng.randint(1, H) for _ in range(n)]
    boxes = [rng.randint(1, H) for _ in range(m)]
    want = brute_force(boxes, wh)
    for f in ALL[1:]:
        assert f(list(boxes), list(wh)) == want, (f.__name__, boxes, wh)
print("1500 bộ ngẫu nhiên khớp mô phỏng: OK")

1500 bộ ngẫu nhiên khớp mô phỏng: OK


In [17]:
# Kiểm tra hàm không làm hỏng dữ liệu (bản solution_copy truyền bản sao)
boxes0, wh0 = [3, 5, 5, 2], [2, 1, 3, 4, 5]
_ = solution_copy(boxes0, wh0)
assert boxes0 == [3, 5, 5, 2] and wh0 == [2, 1, 3, 4, 5]

# Test lớn đúng ràng buộc
rng = random.Random(15)
big = [
    ([rng.randint(1, 10**9) for _ in range(10**5)], [rng.randint(1, 10**9) for _ in range(10**5)]),
    ([1] * 10**5, [10**9] * 10**5),
    ([10**9] * 10**5, [1] + [10**9] * (10**5 - 2) + [1]),
]
for boxes, wh in big:
    t0 = time.perf_counter()
    res = [f(list(boxes), list(wh)) for f in [max_boxes_eff, max_boxes_two_pointers, solution_copy]]
    assert len(set(res)) == 1
    print(f"kết quả {res[0]:>6}  ({time.perf_counter() - t0:.3f}s cho 3 cách)")
assert max_boxes_two_pointers([1] * 10**5, [10**9] * 10**5) == 10**5
assert max_boxes_two_pointers([10**9] * 10**5, [1] + [10**9] * (10**5 - 2) + [1]) == 0

kết quả    429  (0.135s cho 3 cách)
kết quả 100000  (0.073s cho 3 cách)
kết quả      0  (0.078s cho 3 cách)


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ xét đẩy từ **trái** (lời giải của bài I — LC 1564)

In [18]:
def wrong_left_only(boxes, warehouse):
    eff, cur = [], inf
    for h in warehouse:
        cur = min(cur, h)
        eff.append(cur)                                  # SAI: bỏ qua phía phải
    eff.sort()
    ans = i = 0
    for x in sorted(boxes):
        while i < len(eff) and eff[i] < x:
            i += 1
        if i == len(eff):
            break
        ans, i = ans + 1, i + 1
    return ans

b, w = [1, 2, 2, 3, 4], [3, 4, 1, 2]
print("sai:", wrong_left_only(b, w), "| đúng:", solution_copy(b, w))

sai: 3 | đúng: 4


### Lỗi 2 — Dùng `min(left, right)` thay vì `max`

Hộp chỉ cần đi được từ **một** trong hai phía, nên lấy phía tốt hơn (`max`).

In [19]:
def wrong_min_side(boxes, warehouse):
    n = len(warehouse)
    left, right = [inf] * n, [inf] * n
    for i in range(1, n):
        left[i] = min(left[i - 1], warehouse[i - 1])
    for i in range(n - 2, -1, -1):
        right[i] = min(right[i + 1], warehouse[i + 1])
    eff = sorted(min(warehouse[i], min(left[i], right[i])) for i in range(n))   # SAI
    ans = i = 0
    for x in sorted(boxes):
        while i < n and eff[i] < x:
            i += 1
        if i == n:
            break
        ans, i = ans + 1, i + 1
    return ans

b, w = [3, 5, 5, 2], [2, 1, 3, 4, 5]
print("sai:", wrong_min_side(b, w), "| đúng:", solution_copy(b, w))

sai: 0 | đúng: 3


### Lỗi 3 — Quên chặn bởi chính chiều cao phòng `warehouse[i]`

`left[i]`, `right[i]` **không gồm** phòng `i`. Nếu quên `min(warehouse[i], …)` thì một phòng rất thấp ở giữa lại được coi là cao.

In [20]:
def wrong_no_cap(boxes, warehouse):
    n = len(warehouse)
    left, right = [inf] * n, [inf] * n
    for i in range(1, n):
        left[i] = min(left[i - 1], warehouse[i - 1])
    for i in range(n - 2, -1, -1):
        right[i] = min(right[i + 1], warehouse[i + 1])
    eff = sorted(max(left[i], right[i]) for i in range(n))                    # SAI: thiếu min với warehouse[i]
    ans = i = 0
    for x in sorted(boxes):
        while i < n and eff[i] < x:
            i += 1
        if i == n:
            break
        ans, i = ans + 1, i + 1
    return ans

b, w = [3, 3, 3], [5, 1, 5]
print("sai:", wrong_no_cap(b, w), "| đúng:", solution_copy(b, w))

sai: 3 | đúng: 2


### Lỗi 4 — Cách 3 nhưng duyệt hộp **tăng dần**

Hộp nhỏ chiếm mất phòng cửa (cao), khiến hộp lớn đến sau không còn chỗ.

In [21]:
def wrong_ascending(boxes, warehouse):
    l, r, ans = 0, len(warehouse) - 1, 0
    for x in sorted(boxes):                              # SAI: phải là giảm dần
        if l > r:
            break
        if x <= warehouse[l]:
            l += 1
            ans += 1
        elif x <= warehouse[r]:
            r -= 1
            ans += 1
    return ans

b, w = [1, 5], [5, 1]
print("sai:", wrong_ascending(b, w), "| đúng:", solution_copy(b, w))

sai: 1 | đúng: 2


### Lỗi 5 — Tham lam ghép nhưng tăng con trỏ phòng cả khi hộp không vừa

Khi hộp `x` lớn hơn `eff[i]`, phải bỏ **phòng** (tăng `i`) chứ không phải bỏ hộp; viết vòng lặp song song tăng cả hai sẽ bỏ lỡ.

In [22]:
def wrong_parallel(boxes, warehouse):
    eff = sorted(effective_heights(warehouse))
    b = sorted(boxes)
    ans = 0
    for x, e in zip(b, eff):                             # SAI: ghép theo cùng vị trí
        if x <= e:
            ans += 1
    return ans

b, w = [2, 2], [3, 3, 1]
print("eff sắp:", sorted(effective_heights(w)), "| boxes sắp:", sorted(b))
print("sai:", wrong_parallel(b, w), "| đúng:", solution_copy(b, w))

eff sắp: [1, 3, 3] | boxes sắp: [2, 2]
sai: 1 | đúng: 2


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Vật đi qua hành lang, bị chặn bởi chỗ hẹp nhất"** ⇒ sức chứa thực = **min tiền tố** (một phía) hoặc **max(min tiền tố, min hậu tố)** (hai phía).
- Sau khi tính sức chứa hiệu dụng, bài toán thành **ghép cặp đơn giản**: sắp cả hai, tham lam "nhỏ nhất còn vừa".
- Hoặc tư duy ngược: **hộp lớn trước**, đặt ở cửa (ngoài cùng); hộp không vừa cửa nào thì bỏ.
- Cẩn thận chi tiết: `left/right` có gồm phòng `i` hay không; luôn kẹp với `warehouse[i]`.
- Kiểm tra nhanh: `[3,3,3]` vào `[5,1,5]` → `2`; `[1,1,1]` vào `[5,1,5]` → `3`; bài I trên ví dụ 1 cho `3` còn bài II cho `4`.

## 15. Biến thể của bài

1. **LC 1564 — Put Boxes Into the Warehouse I:** chỉ đẩy từ trái ⇒ `eff[i] = min(wh[0..i])` (bài tập 4).
2. **Trả về kế hoạch đặt hộp** (phòng, phía, thứ tự đẩy) — bài tập 5.
3. **Mỗi phòng chứa được nhiều hộp chồng lên nhau** (tổng chiều cao ≤ phòng): trở thành bài toán đóng thùng, khó hơn nhiều.
4. **Hộp có cả chiều rộng**: phòng có chiều rộng khác nhau ⇒ hai chiều ràng buộc.
5. **LC 455 — Assign Cookies, LC 881 — Boats to Save People:** cùng khung "sắp + ghép tham lam/hai con trỏ".

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng + vét cạn | thử mọi hộp, mọi phía, ghi nhớ trạng thái | hàm mũ | hàm mũ | oracle cho dữ liệu rất nhỏ |
| 2. Chiều cao hiệu dụng | `eff = min(wh, max(left, right))`, sắp + ghép | `O(n log n + m log m)` | `O(n)` | lời giải chuẩn, dễ chứng minh |
| 2'. `eff` giảm dần | ghép hộp lớn nhất với phòng tốt nhất | như trên | `O(n)` | cầu nối tới cách 3 |
| 3. Hai con trỏ | hộp giảm dần, đặt ở cửa trái/phải nếu vừa | `O(m log m + n)` | `O(1)` | code ngắn, không cần mảng phụ |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `maxBoxesInWarehouse([1, 2, 3], [1, 2, 3, 4])` trả về bao nhiêu?

**Bài 2 (tính tay):** Tính mảng chiều cao hiệu dụng `eff` cho `warehouse = [5, 2, 6, 3, 4]`.

**Bài 3 (điền TODO):** Viết `ex3_effective(warehouse)` trả về `eff` **không** dùng mảng `left/right` mà dùng min tiền tố và min hậu tố **gồm cả phòng `i`**.

**Bài 4 (tự viết lại — LC 1564):** Viết `ex4_left_only(boxes, warehouse)` cho phiên bản chỉ đẩy từ trái.

**Bài 5 (mini challenge):** Viết `ex5_plan(boxes, warehouse)` trả về **kế hoạch**: danh sách các bước `(chiều_cao_hộp, phòng, phía)` theo **thứ tự đẩy**, với số bước bằng đáp án tối ưu. Kiểm tra bằng bộ xác thực: mỗi bước, phòng đích và mọi phòng trên đường đi (từ phía đã chọn) còn trống và cao ≥ hộp.

In [23]:
# Bài 3 — khung TODO (chỉ định nghĩa, KHÔNG gọi)
def ex3_effective(warehouse):
    n = len(warehouse)
    L = [0] * n     # L[i] = min(warehouse[0..i])
    R = [0] * n     # R[i] = min(warehouse[i..n-1])
    # TODO: tính L bằng một lượt trái -> phải
    # TODO: tính R bằng một lượt phải -> trái
    # TODO: trả về [max(L[i], R[i]) for i in range(n)]
    return None

In [24]:
# Bài 5 — khung TODO (chỉ định nghĩa, KHÔNG gọi)
def ex5_plan(boxes, warehouse):
    # TODO: tính L, R; phòng i phục vụ từ 'L' nếu L[i] >= R[i], ngược lại 'R'
    # TODO: ghép tham lam hộp nhỏ nhất -> phòng eff nhỏ nhất còn vừa (giữ chỉ số phòng)
    # TODO: thứ tự đẩy: phòng phía trái theo chỉ số giảm dần, phòng phía phải theo chỉ số tăng dần
    return []

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Từ trái: chỉ hộp ≤ 1 qua được phòng 0. (2) Từ phải: phòng 3 nhận ≤ 4, phòng 2 nhận ≤ 3, phòng 1 nhận ≤ 2. (3) `eff = [1,2,3,4]`.

**Bài 2.** (1) `L = [5,2,2,2,2]`. (2) `R = [2,2,3,3,4]`. (3) `eff[i] = max(L[i], R[i])`.

**Bài 3.** (1) `L[0] = warehouse[0]`, `L[i] = min(L[i-1], warehouse[i])`. (2) Tương tự cho `R` từ cuối. (3) Vì `L`, `R` đã gồm phòng `i` nên **không** cần `min` với `warehouse[i]` nữa.

**Bài 4.** (1) `eff[i] = min(wh[0..i])`. (2) Sắp + ghép như cách 2. (3) Hoặc: hộp giảm dần, một con trỏ duyệt phòng từ trái.

**Bài 5.** (1) Lưu `(eff, chỉ số)` để sau khi ghép biết phòng nào. (2) Phía của phòng `i`: `'L'` nếu `L[i] ≥ R[i]`. (3) Đẩy phòng trái theo chỉ số giảm dần (xa cửa trái trước), phòng phải theo chỉ số tăng dần.

## 19. Đáp án tham khảo cho bài tập

In [25]:
# Bài 1
ans1 = solution_copy([1, 2, 3], [1, 2, 3, 4])
print("Bài 1:", ans1)
assert ans1 == 3 == brute_force([1, 2, 3], [1, 2, 3, 4])

# Bài 2
eff2 = effective_heights([5, 2, 6, 3, 4])
print("Bài 2:", eff2)
assert eff2 == [5, 2, 3, 3, 4]
print("Bài 1, 2: OK")

Bài 1: 3
Bài 2: [5, 2, 3, 3, 4]
Bài 1, 2: OK


In [26]:
# Bài 3
def ex3_effective_sol(warehouse):
    n = len(warehouse)
    L, R = [0] * n, [0] * n
    L[0] = warehouse[0]
    for i in range(1, n):
        L[i] = min(L[i - 1], warehouse[i])
    R[-1] = warehouse[-1]
    for i in range(n - 2, -1, -1):
        R[i] = min(R[i + 1], warehouse[i])
    return [max(L[i], R[i]) for i in range(n)]

rng = random.Random(3)
for _ in range(1000):
    wh = [rng.randint(1, 20) for _ in range(rng.randint(1, 15))]
    assert ex3_effective_sol(wh) == effective_heights(wh)
print("Bài 3: OK, ví dụ:", ex3_effective_sol([3, 4, 1, 2]))

Bài 3: OK, ví dụ: [3, 3, 1, 2]


In [27]:
# Bài 4 — LC 1564 (chỉ đẩy từ trái)
def ex4_left_only_sol(boxes, warehouse):
    eff, cur = [], inf
    for h in warehouse:
        cur = min(cur, h)
        eff.append(cur)
    eff.sort()
    ans = i = 0
    for x in sorted(boxes):
        while i < len(eff) and eff[i] < x:
            i += 1
        if i == len(eff):
            break
        ans, i = ans + 1, i + 1
    return ans

def ex4_brute(boxes, warehouse):
    wh = tuple(warehouse)

    @lru_cache(maxsize=None)
    def go(mask, rem):
        best = 0
        for h in set(rem):
            rest = list(rem)
            rest.remove(h)
            p = slide(mask, wh, h, "L")
            if p is not None:
                best = max(best, 1 + go(mask | (1 << p), tuple(rest)))
        return best

    return go(0, tuple(sorted(boxes)))

print("Bài 4:", ex4_left_only_sol([4, 3, 4, 1], [5, 3, 3, 4, 1]), ex4_left_only_sol([1, 2, 2, 3, 4], [3, 4, 1, 2]))
assert ex4_left_only_sol([4, 3, 4, 1], [5, 3, 3, 4, 1]) == 3       # ví dụ quen thuộc của LC 1564
rng = random.Random(4)
for _ in range(800):
    wh = [rng.randint(1, 6) for _ in range(rng.randint(1, 6))]
    bx = [rng.randint(1, 6) for _ in range(rng.randint(1, 6))]
    assert ex4_left_only_sol(bx, wh) == ex4_brute(bx, wh), (bx, wh)
print("Bài 4: OK")

Bài 4: 3 3
Bài 4: OK


In [28]:
# Bài 5 — kế hoạch đặt hộp + bộ xác thực
def ex5_plan_sol(boxes, warehouse):
    n = len(warehouse)
    L, R = [0] * n, [0] * n
    L[0], R[-1] = warehouse[0], warehouse[-1]
    for i in range(1, n):
        L[i] = min(L[i - 1], warehouse[i])
    for i in range(n - 2, -1, -1):
        R[i] = min(R[i + 1], warehouse[i])
    rooms = sorted((max(L[i], R[i]), i) for i in range(n))
    assign = []                                   # (hộp, phòng)
    j = 0
    for x in sorted(boxes):
        while j < n and rooms[j][0] < x:
            j += 1
        if j == n:
            break
        assign.append((x, rooms[j][1]))
        j += 1
    left_part = sorted([(room, x) for x, room in assign if L[room] >= R[room]], reverse=True)
    right_part = sorted([(room, x) for x, room in assign if L[room] < R[room]])
    return [(x, room, "L") for room, x in left_part] + [(x, room, "R") for room, x in right_part]

def validate_plan(plan, warehouse):
    filled = set()
    for h, room, side in plan:
        path = range(0, room + 1) if side == "L" else range(room, len(warehouse))
        if room in filled or any(k in filled or warehouse[k] < h for k in path):
            return False
        filled.add(room)
    return True

plan = ex5_plan_sol([1, 2, 2, 3, 4], [3, 4, 1, 2])
print("Bài 5 — kế hoạch cho ví dụ 1:", plan)
assert validate_plan(plan, [3, 4, 1, 2]) and len(plan) == 4
rng = random.Random(5)
for _ in range(1500):
    wh = [rng.randint(1, 8) for _ in range(rng.randint(1, 8))]
    bx = [rng.randint(1, 8) for _ in range(rng.randint(1, 8))]
    plan = ex5_plan_sol(bx, wh)
    assert validate_plan(plan, wh) and len(plan) == max_boxes_two_pointers(bx, wh), (bx, wh, plan)
    assert sorted(h for h, _, _ in plan) == sorted(h for h in sorted(bx)[:len(plan)])
print("Bài 5: OK")

Bài 5 — kế hoạch cho ví dụ 1: [(1, 2, 'L'), (3, 1, 'L'), (2, 0, 'L'), (2, 3, 'R')]
Bài 5: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
class Solution:
    def maxBoxesInWarehouse(self, boxes, warehouse):
        n = len(warehouse)
        left, right = [inf] * n, [inf] * n
        for i in range(1, n):
            left[i] = min(left[i - 1], warehouse[i - 1])          # min các phòng bên trái (không gồm i)
        for i in range(n - 2, -1, -1):
            right[i] = min(right[i + 1], warehouse[i + 1])        # min các phòng bên phải (không gồm i)
        eff = sorted(min(warehouse[i], max(left[i], right[i])) for i in range(n))
        ans = i = 0
        for x in sorted(boxes):                                   # hộp nhỏ nhất -> phòng nhỏ nhất còn vừa
            while i < n and eff[i] < x:
                i += 1
            if i == n:
                break
            ans, i = ans + 1, i + 1
        return ans

# Hoặc: hộp giảm dần, l/r ở hai cửa: vừa trái -> l+=1; vừa phải -> r-=1; không vừa -> bỏ.
```

**Checklist trước khi submit:**
- [ ] Tên phương thức `maxBoxesInWarehouse(self, boxes, warehouse)`.
- [ ] `eff[i] = min(warehouse[i], max(left[i], right[i]))` — `max` giữa hai phía, `min` với chính phòng.
- [ ] `left[0] = right[-1] = inf` (phòng ở cửa không bị chặn từ phía đó).
- [ ] Ghép: bỏ **phòng** quá thấp, không bỏ hộp nhỏ.
- [ ] Cách hai con trỏ: hộp **giảm dần**, dừng khi `l > r`.
- [ ] Thử nhanh: ví dụ → `4, 3`; `[3,3,3]`/`[5,1,5]` → `2`; `[2]`/`[1]` → `0`.